# DPO: Direct Preference Optimization - 실습 코드 1: DPO 학습 (TRL)

- Tutorial ID: `expand-dpo`
- Tutorial: DPO: Direct Preference Optimization
- Section ID: `expand-dpo-code-1`
- Section: 실습 코드 1: DPO 학습 (TRL)

---

이 노트북은 처음 DPO를 공부하는 분도 따라올 수 있도록, 개념 설명과 상세한 코드 주석을 덧붙여 다시 구성한 버전입니다.

**목차**

0. DPO 이해하기 — 왜 필요한가?
1. 실습 환경 준비
2. 왜 Llama-2-7B 대신 작은 모델을 쓰는가
3. 정책 모델 / 기준 모델 불러오기
4. 토크나이저 살펴보기 + 학습 전 답변 확인
5. 선호도(Preference) 데이터셋 만들기
6. `DPOConfig` — 학습 설정 이해하기
7. `DPOTrainer`로 학습 시작하기
8. 학습 로그 읽는 법
9. 학습 후 답변을 학습 전과 비교하기
10. 모델 저장하기
11. 더 실험해보기 (다음 단계 제안)
12. 정리

**이 노트북에서 배우는 것**

- SFT(지도 미세조정)만으로는 왜 부족한지, DPO가 어떤 문제를 푸는지
- `chosen` / `rejected` 선호 데이터가 실제로 학습 신호(loss)로 바뀌는 과정
- `policy model`(정책 모델)과 `reference model`(기준 모델)이 왜 둘 다 필요한지
- TRL의 `DPOTrainer` / `DPOConfig`를 실제로 다루는 법 (2026년 현재 TRL API 기준으로 검증)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: DPO 학습 (TRL)
#
# 이 노트북은 "정답을 한 번 실행"하는 용도가 아니라,
# DPO의 개념(선호 데이터 -> loss -> 정책 업데이트)이
# 실제 코드에서 한 줄씩 어떻게 구현되는지 추적하기 위한 실습 노트입니다.
#
# 학습 목표:
#   1) chosen/rejected 선호쌍이 학습 신호로 바뀌는 전체 흐름을 이해한다
#   2) policy model과 reference model의 역할 차이를 이해한다
#   3) DPOConfig의 핵심 하이퍼파라미터(beta, learning_rate 등)가
#      학습에 어떤 영향을 주는지 감을 잡는다
#   4) 학습 전/후 모델의 답변이 실제로 어떻게 달라지는지 직접 확인한다
#
# 읽는 순서:
#   1) 마크다운 셀의 개념 설명을 먼저 읽고, 그 다음 코드 셀을 봅니다.
#      (이 노트북은 "코드 먼저, 설명 나중"이 아니라 "설명 먼저, 코드 나중" 순서입니다.)
#   2) 각 코드 셀은 위에서 아래로 순서대로 실행해야 합니다. (Run All 권장)
#   3) print()로 출력되는 중간 결과(토큰, 데이터셋 예시, 학습 로그, 생성된 답변)를
#      눈으로 직접 확인하면서 진행하세요.
#
# 실행 환경 안내:
#   - 이 노트북은 torch/transformers/trl에 의존하므로 Google Colab, 로컬 GPU 환경,
#     또는 서버 환경에서 실행하는 것을 권장합니다.
#   - GPU가 없어도 실행은 되지만(자동으로 CPU로 전환됩니다), 매우 느릴 수 있습니다.
#   - 원본 코드는 meta-llama/Llama-2-7b-hf(70억 파라미터, 접근 승인 필요)를 사용했지만,
#     이 버전은 처음 실습하는 분들의 진입장벽을 낮추기 위해 공개적으로 바로 사용 가능한
#     소형 모델(Qwen/Qwen3-0.6B, 6억 파라미터)로 대체했습니다. 개념은 완전히 동일하며,
#     2장에서 이유를 자세히 설명합니다.
# ============================================================


## 0. DPO 이해하기 — 왜 필요한가?

### 0-1. 지도학습(SFT)만으로는 부족한 이유

언어모델을 "말 잘 듣게" 만드는 첫 단계는 보통 **SFT(Supervised Fine-Tuning, 지도 미세조정)**입니다. "이런 질문엔 이렇게 답해라"라는 정답 예시를 잔뜩 보여주고 그대로 따라 하도록 학습시키는 방식이죠.

그런데 SFT에는 한계가 있습니다. SFT는 "정답 하나"만 보여줄 뿐, **"이 답변이 저 답변보다 왜 더 나은지"는 알려주지 않습니다.** 예를 들어 "친구가 힘들어해요, 위로해주세요"라는 요청에 대해 아래 두 답변을 비교해봅시다.

- A: "힘내세요."
- B: "많이 힘드셨겠어요. 무슨 일이 있었는지 편하게 이야기해주시면 같이 고민해볼게요."

둘 다 "틀린 답"은 아니지만, B가 더 낫다는 감각을 SFT만으로 모델에게 전달하기는 어렵습니다. 이럴 때 필요한 것이 **사람의 선호도(preference)를 직접 학습에 반영하는 방법**이며, 이런 방법들을 통틀어 **RLHF(Reinforcement Learning from Human Feedback, 인간 피드백 기반 강화학습)**라고 부릅니다.

### 0-2. 전통적인 RLHF: Reward Model + PPO

원래 RLHF는 두 단계로 이루어집니다.

1. **Reward Model(보상 모델) 학습**: 사람이 "A와 B 중 뭐가 더 나아?"라고 매긴 선호 데이터를 가지고, 답변에 점수를 매기는 **별도의 채점 모델**을 하나 더 학습시킵니다.
2. **PPO(Proximal Policy Optimization)로 정책 학습**: 이렇게 학습된 채점 모델(Reward Model)을 이용해서, 언어모델이 더 높은 점수를 받는 방향으로 답변을 생성하도록 **강화학습(RL)**으로 조금씩 조정합니다.

이 방식은 실제로 잘 작동하지만 단점도 있습니다.

- 모델을 두 개(Reward Model + 정책 모델) 학습해야 함
- 강화학습 특유의 불안정성 (하이퍼파라미터에 민감하고, 학습이 갑자기 무너지기도 함)
- 구현과 디버깅이 까다로움

### 0-3. DPO의 핵심 아이디어: "채점 모델 없이 바로 비교 학습"

**DPO(Direct Preference Optimization)**는 2023년에 제안된 방법으로, "Reward Model을 따로 학습할 필요 없이, 선호 데이터(어떤 답이 더 나은지)를 언어모델 학습에 직접 반영할 수 있다"는 수학적 사실을 이용합니다. (원 논문 제목이 흥미로운데, "당신의 언어모델은 사실 이미 보상모델이다"라는 뜻입니다.)

즉, "이 답이 저 답보다 낫다"는 비교 데이터를 가지고 언어모델을 곧바로 조금씩 업데이트합니다. Reward Model 학습 단계도, 강화학습 루프도 없이 일반적인 지도학습과 비슷한 방식(loss를 계산하고 역전파)으로 진행되기 때문에 훨씬 단순하고 안정적입니다.

두 파이프라인을 나란히 비교하면 다음과 같습니다.

```
[기존 RLHF]
SFT 모델 → 사람 선호 데이터 → ① Reward Model 학습 → ② PPO로 정책 학습(강화학습) → 최종 모델
                                  (별도 모델을 1개 더 학습)   (강화학습이라 불안정할 수 있음)

[DPO]
SFT 모델 → 사람 선호 데이터 → DPO로 정책 직접 학습 → 최종 모델
                              (Reward Model도, 강화학습 루프도 없음)
```


### 0-4. DPO는 정확히 무엇을 비교하나요?

DPO 학습에는 세 가지 데이터 재료가 필요합니다.

- **프롬프트(prompt)**: 질문이나 지시문
- **선택된 답변(chosen)**: 더 낫다고 평가된 답변
- **거부된 답변(rejected)**: 덜 낫다고 평가된 답변

그리고 모델도 두 개가 등장합니다.

- **정책 모델(policy model)**: 우리가 실제로 학습시켜서 파라미터를 바꿔나갈 모델. 최종적으로 우리가 쓰게 될 모델입니다.
- **기준 모델(reference model)**: 학습을 시작하는 시점의 모델을 그대로 얼려서(파라미터 고정) 복사해둔 것. 정책 모델이 "얼마나 원래 모습에서 벗어났는지" 재는 기준점 역할을 합니다.

기준 모델이 왜 필요할까요? 비유를 들자면, 다이어트를 할 때 "원래 체중"을 기준점으로 두고 거기서 너무 급격하게 벗어나지 않으면서 조금씩 개선해나가는 것과 비슷합니다. 기준 모델이 없다면 정책 모델이 "선택된 답변의 확률을 높이는 데"에만 지나치게 몰두해서, 문법이 깨지거나 같은 말을 반복하는 등 이상하고 부자연스러운 텍스트를 생성하도록 망가질 위험이 있습니다.

DPO는 학습 과정에서 매 스텝마다 대략 다음과 같은 계산을 합니다.

1. 정책 모델이 `chosen` 답변에 부여하는 확률과, 기준 모델이 `chosen` 답변에 부여하는 확률을 비교합니다. (정책 모델이 chosen 쪽을 원래보다 얼마나 더 선호하게 됐는지를 봅니다.)
2. 같은 방식으로 `rejected` 답변에 대해서도 비교합니다.
3. "chosen을 더 선호하게 된 정도"가 "rejected를 더 선호하게 된 정도"보다 커지는 방향으로 파라미터를 업데이트합니다.

이때 두 선호도 변화의 차이가 클수록 loss가 작아지도록 설계되어 있고, **`beta`라는 값이 "기준 모델에서 얼마나 벗어나는 것을 허용할지"를 조절하는 손잡이** 역할을 합니다. `beta`가 작으면 기준 모델에서 과감하게 벗어날 수 있어 효과는 강하지만 불안정해질 위험이 있고, `beta`가 크면 기준 모델 근처에 머무르려는 힘이 강해져서 안전하지만 효과가 약할 수 있습니다. (6장에서 실제 코드로 다시 다룹니다.)

### 0-5. 미니 용어 사전

| 용어 | 뜻 |
|---|---|
| SFT (Supervised Fine-Tuning) | 정답 예시를 보여주고 그대로 따라 하도록 학습하는 지도 미세조정 |
| RLHF | 사람의 피드백(선호도)을 학습에 반영하는 방법들을 통틀어 부르는 말 |
| Reward Model | 답변에 점수를 매기도록 학습된 별도의 채점 모델 (전통적 RLHF에서 사용) |
| PPO | Reward Model의 점수를 기반으로 정책을 강화학습으로 개선하는 알고리즘 |
| DPO | Reward Model 없이, 선호 데이터로 정책 모델을 직접 최적화하는 방법 |
| Policy Model (정책 모델) | 우리가 실제로 학습시키는, 파라미터가 계속 업데이트되는 모델 |
| Reference Model (기준 모델) | 학습 시작 시점의 모델을 얼려서 고정해 둔, 비교 기준용 모델 |
| chosen / rejected | 사람(또는 평가자)이 "더 낫다" / "덜 낫다"고 판단한 답변 쌍 |
| beta (β) | 정책 모델이 기준 모델에서 얼마나 벗어나도 되는지 조절하는 계수 |

> 이 노트북은 수식을 엄밀하게 유도하지는 않습니다. 수식이 궁금하다면 12장의 참고 자료에 있는 원 논문을 확인해보세요. 여기서는 "chosen과 rejected에 대한 선호도 차이를 키우는 방향으로 학습한다"는 감각만 잡고 넘어가도 이후 코드를 이해하는 데 충분합니다.


## 1. 실습 환경 준비

DPO 학습을 위해서는 몇 가지 라이브러리가 필요합니다. 각각이 어떤 역할을 하는지 먼저 간단히 정리하고 넘어가겠습니다.

| 라이브러리 | 역할 |
|---|---|
| `torch` | 딥러닝 연산(텐서 계산, 역전파 등)을 담당하는 핵심 엔진 |
| `transformers` | 사전학습된 언어모델과 토크나이저를 불러오고 다루는 라이브러리 |
| `datasets` | 데이터셋을 만들고 다루기 위한 라이브러리 |
| `trl` | SFT/DPO/PPO 등 언어모델 후처리(post-training) 학습 알고리즘을 구현해둔 라이브러리 (Transformer Reinforcement Learning) |
| `accelerate` | CPU/단일 GPU/여러 GPU 등 다양한 실행 환경 차이를 자동으로 처리해주는 라이브러리 |

아래 셀을 실행해서 필요한 라이브러리를 설치하세요. (Colab이라면 `torch`는 보통 이미 설치되어 있어서 빠르게 끝납니다.)


In [ ]:
# -q : 설치 로그를 간략하게 표시
# -U : 이미 설치되어 있어도 최신 버전으로 업그레이드
#
# 참고: 이 노트북은 2026년 기준 최신 trl(1.x) / transformers(5.x) API를 기준으로 검증되었습니다.
# 만약 trl 구버전(0.11 이하 등)이 이미 설치되어 있으면 일부 인자 이름
# (예: tokenizer= -> processing_class=)이 달라서 에러가 날 수 있으니,
# 아래처럼 최신 버전으로 설치/업그레이드하는 것을 권장합니다.
!pip install -q -U transformers trl datasets accelerate matplotlib


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig
from datasets import Dataset
import matplotlib.pyplot as plt
import trl, transformers, datasets

# 설치된 라이브러리 버전을 확인해둡니다.
# (나중에 코드가 예상과 다르게 동작하면 가장 먼저 버전을 의심해보세요.)
print(f"torch       : {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"trl         : {trl.__version__}")
print(f"datasets    : {datasets.__version__}")

# ------------------------------------------------------------
# GPU 사용 가능 여부와, 사용할 수 있는 최적의 연산 정밀도(dtype)를 자동으로 판단합니다.
#
# 왜 이런 게 필요할까요?
#  - GPU 메모리는 한정되어 있는데, 모델 파라미터를 32비트(float32) 대신
#    16비트(float16 / bfloat16)로 들고 있으면 메모리 사용량을 절반으로 줄일 수 있습니다.
#  - 다만 모든 GPU가 bfloat16을 지원하는 것은 아니어서(예: Colab 무료 T4는 미지원),
#    지원 여부를 먼저 확인한 뒤 적절한 정밀도를 선택해야 합니다.
#  - GPU가 아예 없다면(CPU 전용 환경) float32로 진행합니다. (매우 느릴 수 있습니다)
# ------------------------------------------------------------
use_cuda = torch.cuda.is_available()
use_bf16 = use_cuda and torch.cuda.is_bf16_supported()
use_fp16 = use_cuda and not use_bf16

if use_bf16:
    dtype = torch.bfloat16
elif use_fp16:
    dtype = torch.float16
else:
    dtype = torch.float32

print()
print(f"GPU 사용 가능: {use_cuda}")
if use_cuda:
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
print(f"선택된 연산 정밀도(dtype): {dtype}")

if not use_cuda:
    print("\n[주의] GPU가 감지되지 않았습니다. CPU로도 실행은 되지만 매우 느릴 수 있습니다.")
    print("Colab이라면 [런타임] -> [런타임 유형 변경] -> [하드웨어 가속기]를 GPU로 설정해보세요.")


## 2. 왜 Llama-2-7B 대신 작은 모델을 쓰나요?

원본 실습 코드는 `meta-llama/Llama-2-7b-hf`를 사용했습니다. 개념을 이해하는 데는 전혀 문제가 없지만, **처음 실습해보는 입장에서는 두 가지 걸림돌**이 있습니다.

1. **접근 승인이 필요합니다.** Llama-2는 Meta의 라이선스에 동의하고 HuggingFace 계정으로 접근 승인을 받아야만 다운로드할 수 있는 "gated 모델"입니다. 승인에는 시간이 걸릴 수 있고, 승인 후에도 HuggingFace 토큰으로 로그인하는 절차가 추가로 필요합니다.
2. **메모리가 많이 필요합니다.** 70억(7B) 개 파라미터 모델을 정책 모델 + 기준 모델로 두 벌 올리면, 최소 수십 GB의 GPU 메모리가 필요합니다. Colab 무료 GPU로는 버겁습니다.

그래서 이 노트북에서는 **`Qwen/Qwen3-0.6B`** 모델을 사용합니다.

- 파라미터가 약 6억 개로, Llama-2-7B의 1/10 수준이라 무료 Colab GPU에서도 정책 모델과 기준 모델을 동시에 올릴 수 있습니다.
- 별도의 접근 승인이나 로그인 없이 누구나 바로 내려받을 수 있는 공개 모델입니다.
- 이미 지시(instruction)를 따르도록 학습되어 있어서, 우리가 만들 선호 데이터로 DPO를 적용했을 때 답변 스타일이 바뀌는 모습을 바로 관찰할 수 있습니다.

> 개념은 모델 크기와 무관하게 동일합니다. 아래 코드의 `model_name` 변수만 `"meta-llama/Llama-2-7b-hf"`나 다른 모델 이름으로 바꾸면, (충분한 GPU 메모리와 HuggingFace 접근 권한이 있다는 전제 하에) 이 노트북의 나머지 코드를 그대로 재사용할 수 있습니다.


## 3. 정책 모델(policy model)과 기준 모델(reference model) 불러오기

앞서 0장에서 설명했듯, DPO 학습에는 모델이 두 개 필요합니다.

- **정책 모델(`model`)**: 실제로 파라미터가 업데이트되는, 우리가 학습시킬 모델
- **기준 모델(`ref_model`)**: 학습 시작 시점 그대로 얼려서(파라미터 고정) 복사해 둔 모델. 정책 모델이 얼마나 원래 모습에서 벗어났는지 재는 "기준 눈금" 역할을 합니다.

코드 상에서는 **완전히 동일한 모델을 두 번 불러와서**, 하나는 학습용(`model`)으로, 다른 하나는 비교 기준용(`ref_model`)으로 사용합니다. `ref_model`은 학습 내내 파라미터가 바뀌지 않고 그대로 유지됩니다.

> **참고**: 최신 버전의 TRL `DPOTrainer`에서는 `ref_model`을 아예 생략해도 됩니다. 생략하면 TRL이 자동으로 `model`을 복사해서 기준 모델을 만들어줍니다. 특히 LoRA 같은 PEFT(Parameter-Efficient Fine-Tuning) 기법을 쓸 때는 어댑터(adapter)를 잠깐 꺼서 기준 모델 역할을 대신하기 때문에, 별도의 모델을 메모리에 두 벌 올릴 필요조차 없습니다. (11장에서 다시 다룹니다.) 다만 이 노트북에서는 "기준 모델이 실제로 무엇인지" 눈으로 확인할 수 있도록 명시적으로 두 개를 불러오겠습니다.


In [ ]:
model_name = "Qwen/Qwen3-0.6B"  # 사용할 사전학습 모델 이름 (HuggingFace Hub 기준)

# 토크나이저: 텍스트 <-> 토큰(숫자) 변환을 담당합니다. (자세한 내용은 바로 다음 섹션에서 확인합니다)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 정책 모델(policy model): 이번 실습에서 실제로 파라미터가 업데이트될 모델입니다.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=dtype,  # 위에서 자동으로 결정한 연산 정밀도를 사용 (구버전 transformers라면 torch_dtype 인자를 대신 사용하세요)
)

# 기준 모델(reference model): model과 완전히 동일하게 시작하지만,
# 이후 학습 과정에서 파라미터를 업데이트하지 않고 "그대로" 둡니다.
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=dtype,
)

# GPU가 있다면 두 모델을 GPU 메모리로 옮깁니다.
if use_cuda:
    model = model.to("cuda")
    ref_model = ref_model.to("cuda")

print("모델 로드 완료")
print(f"정책 모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"모델이 올라간 장치: {model.device}")


## 4. 토크나이저(Tokenizer) 살펴보기

언어모델은 문자 그대로의 "텍스트"를 이해하지 못합니다. 대신 텍스트를 정수 ID로 이루어진 **토큰(token)** 시퀀스로 바꿔서 처리하는데, 이 변환을 담당하는 것이 토크나이저입니다.

- `tokenizer.tokenize(text)`: 텍스트를 토큰(부분 단어) 문자열 리스트로 쪼갭니다.
- `tokenizer.encode(text)`: 텍스트를 토큰 ID(정수) 리스트로 바꿉니다.
- `tokenizer.decode(ids)`: 토큰 ID 리스트를 다시 사람이 읽을 수 있는 텍스트로 되돌립니다.

아래 셀에서 짧은 문장 하나로 이 과정을 직접 확인해봅시다.


In [ ]:
sample_text = "가을을 주제로 짧은 시를 써줘"

tokens = tokenizer.tokenize(sample_text)    # 텍스트 -> 토큰 문자열
token_ids = tokenizer.encode(sample_text)   # 텍스트 -> 토큰 ID(정수)
decoded_text = tokenizer.decode(token_ids)  # 토큰 ID -> 다시 텍스트로

print(f"원본 텍스트   : {sample_text}")
print(f"토큰 개수     : {len(tokens)}")
print(f"토큰 문자열   : {tokens}")
print(f"토큰 ID       : {token_ids}")
print(f"복원된 텍스트 : {decoded_text}")


### 4-1. 학습 전, 지금 모델은 어떻게 답할까요?

DPO를 적용하기 전, 지금 상태의 모델이 우리 데이터셋에 있는 질문에 어떻게 답하는지 먼저 확인해두겠습니다. 이렇게 미리 저장해두면, 학습이 끝난 뒤 "정말 뭔가 달라졌는지"를 직접 비교할 수 있습니다.

먼저 답변을 생성하는 함수를 하나 만들어두겠습니다. (9장의 학습 후 비교에서도 재사용합니다.)


In [ ]:
def generate_answer(target_model, prompt, max_new_tokens=80):
    """주어진 모델(target_model)이 prompt에 대해 생성하는 답변을 문자열로 반환합니다."""

    target_model.eval()  # 평가(추론) 모드로 전환합니다. (학습 때만 필요한 dropout 등을 끕니다)

    # 텍스트를 토큰 ID로 바꾸고, 모델과 같은 장치(CPU/GPU)로 옮깁니다.
    inputs = tokenizer(prompt, return_tensors="pt").to(target_model.device)

    # 추론 중에는 기울기(gradient)를 계산할 필요가 없으므로 torch.no_grad()로 꺼서
    # 계산량과 메모리 사용량을 줄입니다.
    with torch.no_grad():
        output_ids = target_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # 매번 같은 결과가 나오도록 결정적으로(greedy) 생성합니다.
            pad_token_id=tokenizer.eos_token_id,
        )

    # output_ids에는 "입력 프롬프트 + 새로 생성된 부분"이 함께 들어있으므로,
    # 새로 생성된 부분만 잘라냅니다.
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# 이후 데이터셋에도 사용할 질문 중 하나로 미리 테스트해봅니다.
demo_prompt = "가을을 주제로 짧은 시를 써줘"

answer_before = generate_answer(model, demo_prompt)

print(f"[질문] {demo_prompt}")
print(f"[학습 전 답변]\n{answer_before}")


## 5. 선호도(Preference) 데이터셋 만들기

DPO 학습에 쓰이는 데이터는 SFT용 데이터와 조금 다릅니다. SFT는 "이 질문에는 이 정답"처럼 (질문, 정답) 쌍만 있으면 되지만, DPO는 **같은 질문에 대해 상대적으로 비교되는 두 개의 답변**이 필요합니다.

각 데이터는 세 개의 필드로 이루어집니다.

- `prompt`: 질문/지시문
- `chosen`: 더 낫다고 평가된 답변
- `rejected`: 덜 낫다고 평가된 답변

예를 들어 아래 한 쌍을 봅시다.

> **prompt**: "머신러닝이 뭔지 쉽게 설명해줘"
>
> **chosen**: "머신러닝은 사람이 규칙을 일일이 정해주는 대신, 컴퓨터가 많은 데이터를 보면서 스스로 규칙(패턴)을 찾아내게 하는 방법이에요. ..."
>
> **rejected**: "머신러닝은 컴퓨터가 똑똑해지는 기술이에요."

`chosen`이 절대적으로 "완벽한 정답"이라서가 아니라, `rejected`보다 **상대적으로** 더 정확하고 이해하기 쉽기 때문에 선택되었다는 점이 핵심입니다. DPO는 이렇게 상대적인 비교를 학습 신호로 사용합니다.

아래에서는 이해를 돕기 위해 8개의 예시를 직접 만들어보겠습니다. 실제 프로덕션 환경에서는 보통 수천~수만 개의, 사람이 직접(혹은 다른 모델의 도움을 받아) 평가한 데이터를 사용하지만, 여기서는 개념을 눈으로 확인하기 위해 소량의 데이터로 진행합니다. 모든 `rejected` 답변은 "성의 없이 짧고 도움이 안 되는 답변"이라는 공통점을 가지도록 만들어서, chosen/rejected의 차이가 무엇을 학습시키는 신호인지 명확히 보이도록 했습니다.


In [ ]:
data = {
    "prompt": [
        "가을을 주제로 짧은 시를 써줘",
        "머신러닝이 뭔지 쉽게 설명해줘",
        "파이썬에서 리스트(list)랑 튜플(tuple)이 어떻게 다른가요?",
        "친구가 약속을 자꾸 어겨서 너무 속상해. 어떡하지?",
        "환불 요청 이메일을 어떻게 써야 할지 모르겠어요. 도와주세요.",
        "오늘 저녁 뭐 먹을지 추천해줘",
        "이력서 자기소개서를 잘 쓰는 팁을 알려줘",
        "발표를 앞두고 너무 긴장돼. 조언 좀 해줘.",
    ],
    "chosen": [
        "노랗게 물든 은행잎이\n바람 따라 살랑살랑 춤을 추고,\n따뜻한 차 한 잔에\n가을 오후가 조용히 익어간다.",
        "머신러닝은 사람이 규칙을 일일이 정해주는 대신, 컴퓨터가 많은 데이터를 보면서 스스로 규칙(패턴)을 찾아내게 하는 방법이에요. 예를 들어 고양이 사진 수천 장을 보여주면, 컴퓨터가 '고양이는 대략 이런 특징을 가졌구나' 하는 패턴을 스스로 익히는 거죠.",
        "가장 큰 차이는 '변경 가능 여부'예요. 리스트(list)는 만든 뒤에도 값을 추가·삭제·수정할 수 있지만(가변), 튜플(tuple)은 한 번 만들면 값을 바꿀 수 없어요(불변). 그래서 값이 바뀔 일이 없는 데이터는 튜플로, 계속 값이 변하는 목록은 리스트로 쓰는 경우가 많아요.",
        "그런 일이 반복되면 정말 서운하고 지치죠. 감정이 격해지지 않은 때를 골라서 '네가 약속을 못 지킬 때마다 나는 이런 기분이 들어'처럼 비난보다는 내 감정을 중심으로 솔직하게 이야기해보는 게 도움이 될 수 있어요.",
        "간단한 예시를 드릴게요.\n\n제목: [주문번호 12345] 환불 요청 드립니다\n\n안녕하세요, 담당자님. 지난주 구매한 상품(주문번호: 12345)에 문제가 있어 환불을 요청드리고자 합니다. 절차를 안내해 주시면 감사하겠습니다.",
        "날씨가 쌀쌀하다면 얼큰한 김치찌개나 순두부찌개 어때요? 좀 더 가볍게 먹고 싶다면 소불고기덮밥도 좋은 선택이에요. 둘 다 재료가 간단해서 집에서 만들기도 어렵지 않아요.",
        "세 가지만 기억하세요. 첫째, 첫 문장에 지원 직무와 관련된 핵심 강점을 바로 제시하세요. 둘째, '책임감이 강함' 같은 추상적 표현 대신 구체적인 성과나 숫자로 증명하세요. 셋째, 회사가 왜 당신을 뽑아야 하는지로 마무리하세요.",
        "긴장되는 건 자연스러운 반응이에요. 발표 직전에 천천히 심호흡을 몇 번 하면 심박수를 가라앉히는 데 도움이 되고, 첫 문장만이라도 완벽히 외워두면 시작할 때 훨씬 편해져요. 청중도 완벽한 발표보다 진심이 담긴 발표를 더 좋아한다는 걸 기억하세요.",
    ],
    "rejected": [
        "가을은 낙엽이 지고 날씨가 선선해지는 계절입니다.",
        "머신러닝은 컴퓨터가 똑똑해지는 기술이에요.",
        "둘이 비슷해서 아무거나 써도 큰 상관없어요.",
        "그런 친구는 그냥 손절하세요.",
        "그냥 환불해달라고 쓰면 돼요.",
        "아무거나 드세요.",
        "성의있게 잘 쓰면 됩니다.",
        "긴장하지 말고 그냥 잘하세요.",
    ],
}

dataset = Dataset.from_dict(data)

print(f"데이터셋 크기: {len(dataset)}개")
print()
print("=== 첫 번째 예시 ===")
print(f"prompt  : {dataset[0]['prompt']}")
print(f"chosen  : {dataset[0]['chosen']}")
print(f"rejected: {dataset[0]['rejected']}")


## 6. `DPOConfig`: 학습 설정 이해하기

`DPOConfig`는 학습에 필요한 여러 설정값을 모아두는 곳입니다. 처음 보면 옵션이 많아 보이지만, 이번 실습에서 사용할 핵심 옵션만 추려서 하나씩 설명하겠습니다.

| 옵션 | 의미 |
|---|---|
| `output_dir` | 학습 중간 체크포인트와 최종 결과물이 저장될 폴더 경로 |
| `beta` | **DPO의 핵심 하이퍼파라미터.** 정책 모델이 기준 모델에서 얼마나 벗어나도 되는지를 조절합니다. 값이 **작을수록**(예: 0.01) 기준 모델에서 과감하게 벗어나며 선호도를 강하게 반영하지만 불안정해질 위험이 커지고, 값이 **클수록**(예: 0.5) 기준 모델 근처에 머무르려는 힘이 강해져 안전하지만 효과가 약할 수 있습니다. 보통 0.1~0.5 사이 값을 많이 사용합니다. |
| `per_device_train_batch_size` | GPU(또는 CPU) 한 개가 한 번에 처리하는 데이터 개수 |
| `learning_rate` | 파라미터를 한 번에 얼마나 크게 업데이트할지 결정하는 값. DPO는 이미 어느 정도 학습이 된 모델을 "미세하게" 조정하는 것이므로, 처음부터 학습하는 SFT보다 훨씬 작은 값(대략 1e-7~1e-5)을 사용합니다. 너무 크면 모델이 순식간에 망가질 수 있습니다. |
| `num_train_epochs` | 전체 데이터셋을 몇 번 반복해서 학습할지 |
| `logging_steps` | 몇 스텝마다 학습 로그(loss, reward 등)를 출력할지 |
| `max_length` | 프롬프트+답변을 합쳐 최대 몇 토큰까지 사용할지. 이보다 길면 잘립니다(truncate). |
| `bf16` / `fp16` | 16비트 연산 정밀도 사용 여부 (위에서 자동으로 판단한 값을 그대로 사용합니다) |
| `report_to` | 학습 로그를 어디로 보낼지(W&B 등 외부 서비스). 이번 실습에서는 `"none"`으로 꺼둡니다. |

> **왜 데이터가 8개뿐인데 배치 크기와 에폭 수를 이렇게 정했나요?** 데이터가 매우 적기 때문에 `per_device_train_batch_size=2`로 작게 잡아 한 에폭에 4번(8÷2)의 업데이트가 일어나도록 하고, `num_train_epochs=5`로 여러 번 반복해서 총 20번의 업데이트를 거치도록 했습니다. 그래야 8장에서 그릴 그래프에서 변화 추세를 눈으로 확인할 수 있을 만큼의 학습 스텝이 확보됩니다. 실제 프로젝트에서는 데이터가 훨씬 많으므로 에폭 수는 보통 1~3 정도로 충분합니다.


In [ ]:
training_args = DPOConfig(
    output_dir="./dpo-model",       # 결과물이 저장될 폴더

    beta=0.1,                       # 기준 모델로부터의 이탈 허용 정도 (0.1은 흔히 쓰이는 값)

    per_device_train_batch_size=2,  # 데이터가 8개뿐이므로 작은 배치 크기를 사용
    num_train_epochs=5,             # 데이터가 적으므로 여러 번 반복해서 변화를 관찰

    learning_rate=5e-6,             # SFT보다 훨씬 작은 학습률

    logging_steps=1,                # 매 스텝마다 로그를 출력 (데이터가 적어 촘촘히 관찰 가능)
    max_length=512,                 # 우리 데이터는 짧으므로 512토큰이면 충분

    bf16=use_bf16,                  # 1장에서 자동으로 판단한 정밀도 설정을 그대로 사용
    fp16=use_fp16,

    report_to="none",               # 실습 목적이므로 외부 로깅 서비스는 사용하지 않음
)

print("DPOConfig 생성 완료")
print(f"beta={training_args.beta}, learning_rate={training_args.learning_rate}, "
      f"epochs={training_args.num_train_epochs}, batch_size={training_args.per_device_train_batch_size}")


## 7. `DPOTrainer`로 학습 시작하기

이제 지금까지 준비한 모든 재료(정책 모델, 기준 모델, 설정, 데이터셋)를 `DPOTrainer`에 전달해서 실제 학습을 시작합니다.

한 가지 꼭 짚고 넘어갈 부분이 있습니다. **예전 버전의 TRL에서는 토크나이저를 전달할 때 `tokenizer=`라는 인자를 사용했지만, 현재 버전(TRL 0.12 이상)에서는 이 인자 이름이 `processing_class=`로 바뀌었습니다.** (텍스트 전용 토크나이저뿐 아니라 이미지까지 함께 다루는 프로세서도 받을 수 있도록 이름을 일반화한 것입니다.) 오래된 코드나 블로그 글을 참고할 때 `tokenizer=`를 그대로 쓰면 `TypeError: DPOTrainer.__init__() got an unexpected keyword argument 'tokenizer'` 같은 에러가 발생하니 주의하세요. (실제로 이 노트북을 준비하면서 직접 재현하고 확인한 내용입니다.)


In [ ]:
trainer = DPOTrainer(
    model=model,              # 학습시킬 정책 모델
    ref_model=ref_model,      # 비교 기준이 되는, 파라미터가 고정된 기준 모델
    args=training_args,       # 6장에서 만든 학습 설정
    train_dataset=dataset,    # 5장에서 만든 선호도 데이터셋

    # 주의: TRL 0.12 이상에서는 tokenizer= 대신 processing_class= 를 사용합니다.
    processing_class=tokenizer,
)

# 실제 학습을 시작합니다.
# 데이터가 8개뿐이고 모델도 작기 때문에, GPU 환경에서는 1분 이내로 끝나는 경우가 많습니다.
# (CPU 환경이라면 훨씬 오래 걸릴 수 있습니다.)
trainer.train()


## 8. 학습 로그 읽는 법

`trainer.train()`을 실행하면 스텝마다 여러 지표가 출력됩니다. 처음 보면 낯선 이름들이라 헷갈릴 수 있으니 하나씩 정리하겠습니다.

| 지표 | 의미 | 학습이 잘 되고 있다면 |
|---|---|---|
| `loss` | DPO loss 값 | 전반적으로 감소 |
| `rewards/chosen` | 정책 모델이 `chosen` 답변을 기준 모델보다 얼마나 더 선호하게 됐는지 (로그 확률 비율 기반) | 증가 |
| `rewards/rejected` | 정책 모델이 `rejected` 답변을 기준 모델보다 얼마나 더/덜 선호하게 됐는지 | 감소 (또는 chosen보다 덜 오름) |
| `rewards/margins` | `rewards/chosen` - `rewards/rejected`, 즉 두 선호도의 차이 | 증가 (0보다 커지고, 계속 커질수록 좋은 신호) |
| `rewards/accuracies` | 미니배치 안에서 "chosen의 reward가 rejected보다 실제로 높았던" 비율 | 1.0(=100%)에 가까워짐 |

이 지표들은 `trainer.state.log_history`에 스텝별로 쌓여 있습니다. 아래에서 이를 그래프로 그려 눈으로 확인해보겠습니다.


In [ ]:
# trainer.state.log_history: 학습 중 logging_steps마다 기록된 지표들의 리스트입니다.
log_history = trainer.state.log_history

# 각 로그 딕셔너리에 해당 키가 있을 때만 값을 모읍니다.
# (학습 종료 시 나오는 요약 로그처럼 일부 키가 없는 항목도 섞여 있을 수 있기 때문입니다.)
steps             = [log["step"] for log in log_history if "loss" in log]
loss_values       = [log["loss"] for log in log_history if "loss" in log]
chosen_rewards    = [log["rewards/chosen"] for log in log_history if "rewards/chosen" in log]
rejected_rewards  = [log["rewards/rejected"] for log in log_history if "rewards/rejected" in log]
margins           = [log["rewards/margins"] for log in log_history if "rewards/margins" in log]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(steps, loss_values, marker="o", color="tab:red")
axes[0].set_title("Loss (낮아질수록 좋음)")
axes[0].set_xlabel("step")
axes[0].grid(alpha=0.3)

axes[1].plot(steps, chosen_rewards, marker="o", label="rewards/chosen", color="tab:blue")
axes[1].plot(steps, rejected_rewards, marker="o", label="rewards/rejected", color="tab:orange")
axes[1].plot(steps, margins, marker="o", label="rewards/margins", color="tab:green", linestyle="--")
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_title("Rewards (chosen은 위로, rejected는 아래로 갈수록 좋음)")
axes[1].set_xlabel("step")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 9. 학습 후 답변을 학습 전과 비교하기

이제 학습이 끝났으니, 4-1에서 저장해둔 "학습 전" 답변과 같은 질문에 대한 "학습 후" 답변을 나란히 비교해봅시다.


In [ ]:
answer_after = generate_answer(model, demo_prompt)

print(f"[질문] {demo_prompt}")
print()
print(f"[학습 전]\n{answer_before}")
print()
print(f"[학습 후]\n{answer_after}")


> **참고**: 우리 데이터셋은 8개뿐이고 5 에폭만 학습했기 때문에, 극적인 변화보다는 스타일이 조금씩 우리가 `chosen`으로 준 방향(더 구체적이고 정성스러운 답변)에 가까워지는 정도의 변화를 관찰하게 될 가능성이 높습니다. 데이터가 적을수록 모델이 "우연히 학습 데이터와 비슷한 질문에서만" 달라지고, 새로운 질문에는 잘 일반화되지 않을 수 있다는 점도 기억해두세요. (이건 결함이 아니라 데이터가 8개뿐인 실습이기 때문에 나타나는 자연스러운 한계입니다.) 아래 셀에서 학습 데이터에 없던 새로운 질문으로도 직접 테스트해보세요.


In [ ]:
# 직접 해보기: 학습 데이터에 없던 새로운 질문으로도 테스트해보세요.
new_prompt = "겨울을 주제로 짧은 시를 써줘"  # 원하는 질문으로 자유롭게 바꿔보세요.

print(f"[질문] {new_prompt}")
print(f"[학습 후 답변]\n{generate_answer(model, new_prompt)}")


## 10. 모델 저장하기

학습된 정책 모델을 나중에 다시 불러와 쓸 수 있도록 저장해둡니다.


In [ ]:
trainer.save_model("./dpo-model/final")
tokenizer.save_pretrained("./dpo-model/final")

print("모델 저장 완료: ./dpo-model/final")
print("나중에 다시 불러올 때는 아래처럼 사용하면 됩니다.")
print('  AutoModelForCausalLM.from_pretrained("./dpo-model/final")')


## 11. 더 실험해보기 (다음 단계 제안)

개념을 익혔다면, 아래와 같은 실험을 직접 해보면서 감각을 더 키워볼 수 있습니다.

1. **`beta` 값을 바꿔보기**: `beta=0.01`처럼 작게 주면 어떻게 될까요? `beta=0.5`처럼 크게 주면요? `rewards/margins`가 얼마나 빠르게, 그리고 얼마나 안정적으로 커지는지 비교해보세요.
2. **`learning_rate`를 바꿔보기**: 너무 크게 주면(`1e-4` 등) 어떤 문제가 생기는지 관찰해보세요. (답변이 이상해지거나 같은 말을 반복하는 등 "망가진" 징후를 볼 수 있습니다.)
3. **실제 공개 선호 데이터셋 사용해보기**: TRL 공식 문서에서 예시로 자주 쓰이는 `trl-lib/ultrafeedback_binarized` 데이터셋을 사용하면 훨씬 규모 있는 학습을 경험할 수 있습니다.

   ```python
   from datasets import load_dataset
   real_dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
   ```

4. **LoRA(PEFT)로 메모리를 아껴서 더 큰 모델 학습해보기**: `peft_config`를 `DPOTrainer`에 전달하면, 모델 전체가 아니라 일부 파라미터(어댑터)만 학습해서 메모리를 크게 아낄 수 있습니다. 이 경우 `ref_model`을 따로 만들 필요도 없습니다(어댑터를 잠깐 꺼서 기준 모델 역할을 대신하기 때문입니다).

   ```python
   from peft import LoraConfig
   trainer = DPOTrainer(
       model=model,
       args=training_args,
       train_dataset=dataset,
       processing_class=tokenizer,
       peft_config=LoraConfig(),
   )
   ```

5. **더 큰 모델로 전환해보기**: `model_name`을 원래 예제의 `"meta-llama/Llama-2-7b-hf"`나 다른 모델로 바꿔보세요. (단, 접근 승인 및 `huggingface-cli login`, 그리고 충분한 GPU 메모리가 필요합니다.)


## 12. 정리

이 노트북에서 다룬 내용을 정리하면 다음과 같습니다.

- SFT는 "정답"만 알려줄 뿐 "더 나은 답"에 대한 감각은 알려주지 못하기 때문에, 사람의 선호도를 반영하는 별도의 학습 단계가 필요합니다.
- 전통적인 RLHF(Reward Model + PPO)는 강력하지만 두 단계 파이프라인과 강화학습의 불안정성이라는 부담이 있습니다.
- **DPO는 별도의 Reward Model 없이, `chosen`/`rejected` 선호쌍을 이용해 정책 모델을 직접 최적화**합니다.
- 정책 모델(`model`)이 실제로 학습되는 동안, 기준 모델(`ref_model`)은 "너무 멀리 벗어나지 않도록" 잡아주는 닻 역할을 하며, 이 균형은 `beta` 값으로 조절됩니다.
- TRL의 `DPOTrainer`/`DPOConfig`를 이용하면 이 과정을 몇 줄의 코드로 실행할 수 있고, `rewards/chosen`, `rewards/rejected`, `rewards/margins` 같은 로그로 학습이 잘 되고 있는지 확인할 수 있습니다.

**참고 자료**

- 원 논문: Rafailov et al., *Direct Preference Optimization: Your Language Model is Secretly a Reward Model* (2023)
- TRL 공식 문서: https://huggingface.co/docs/trl/main/en/dpo_trainer

수고하셨습니다! 다음 실습에서는 더 다양한 선호 최적화 방법(ORPO, KTO 등)이나, 실제 대규모 데이터셋을 활용한 학습을 다뤄볼 수 있습니다.
